# Clinical Score Training

Train on unhealthy clinician-scored recordings; predict 0–4 scores for healthy recordings.


In [ ]:
from pathlib import Path
import csv
import openpyxl
import numpy as np

ROOT = Path.cwd()
required = [ROOT / "Clinical Score.xlsx", ROOT / "data/features/recording_level_features.csv"]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Supply the local clinical workbook and feature table: " + ", ".join(missing))

with (ROOT / "data/features/recording_level_features.csv").open(newline="") as handle:
    rows = list(csv.DictReader(handle))

book = openpyxl.load_workbook(ROOT / "Clinical Score.xlsx", data_only=True, read_only=True)
scores = {}
for sheet in book.worksheets:
    participant = {"A_PD001": "Atypical_PD001"}.get(sheet.title, sheet.title)
    scores[participant] = {
        int(str(row[0]).split(".")[0]): float(row[3])
        for row in sheet.iter_rows(min_row=4, max_row=36, values_only=True)
        if row[0] and row[3] is not None
    }

print(f"Recordings: {len(rows)}")
print("Diagnosis counts:", {d: sum(r["diagnosis"] == d for r in rows) for d in sorted({r["diagnosis"] for r in rows})})
print("Participant counts:", {p: sum(r["participant_id"] == p for r in rows) for p in sorted({r["participant_id"] for r in rows})})
print("Clinician Part III totals:", {p: sum(items.values()) for p, items in scores.items()})

## Features and labels

Task-to-score mapping is provisional.


In [ ]:
TASK_SCORE_ITEM = {
    "Arm Extended Forward": 21, "Finger Tapping": 11, "Finger to Nose": 21,
    "Hand Resting": 6, "Fist Open and Close": 12, "Pronation Supination": 13,
    "Normal Walking": 16, "Quiet Standing": 17, "Sit-to-Stand": 15, "TUG": 15,
}

labeled = []
for source in rows:
    item = TASK_SCORE_ITEM.get(source["task"])
    if source["participant_id"] in scores and item in scores[source["participant_id"]]:
        row = dict(source)
        row["clinical_score"] = scores[source["participant_id"]][item]
        row["mapped_item"] = item
        labeled.append(row)

META = {
    "recording_id", "participant_id", "diagnosis", "session_id", "task", "file_path",
    "file_type", "sensor_side", "sensor_location", "signal_column_used", "quality_status",
    "quality_warnings", "clinical_score", "clinical_score_source", "label_verified",
    "mapped_item", "recording_duration_s", "sampling_rate_hz",
}
numeric_names = []
for name in labeled[0]:
    if name in META:
        continue
    try:
        [float(row[name]) for row in labeled if row[name] != ""]
        numeric_names.append(name)
    except (ValueError, TypeError):
        pass

def matrix(records):
    return np.array([[float(row.get(name, "") or np.nan) for name in numeric_names] for row in records], dtype=float)

X = matrix(labeled)
y = np.array([row["clinical_score"] for row in labeled], dtype=float)
groups = np.array([row["participant_id"] for row in labeled])
healthy = [row for row in rows if row["diagnosis"] == "Healthy"]
X_healthy = matrix(healthy)

print(f"Mapped labels: {len(labeled)} / {len(rows)} recordings")
print("Mapped labels by participant:", {str(p): int(sum(groups == p)) for p in sorted(set(groups))})
print(f"Healthy rows reserved for 0–4 prediction: {len(healthy)}")
print(f"Numeric features: {len(numeric_names)}")

In [ ]:
from collections import Counter

print("Data statistics")
print(f"Total recordings: {len(rows)}")
print(f"Unhealthy labeled: {len(labeled)} | Healthy test: {len(healthy)}")
print(f"Features: {len(numeric_names)}")
print("Diagnosis:", dict(Counter(row["diagnosis"] for row in rows)))
print("Score distribution:", {int(score): int(count) for score, count in Counter(y.astype(int)).items()})
print("Missing feature values:", int(np.isnan(X).sum()), "of", X.size)

In [ ]:
import matplotlib.pyplot as plt

# PCA on imputed, standardized features for a 2D data overview.
all_features = np.vstack([X, X_healthy])
pca_medians = np.zeros(all_features.shape[1])
for index in range(all_features.shape[1]):
    valid = all_features[:, index][np.isfinite(all_features[:, index])]
    pca_medians[index] = np.median(valid) if len(valid) else 0.0
all_features = np.where(np.isfinite(all_features), all_features, pca_medians)
all_features = (all_features - all_features.mean(0)) / np.maximum(all_features.std(0), 1e-12)
_, singular_values, components = np.linalg.svd(all_features, full_matrices=False)
projection = all_features @ components[:2].T
variance = singular_values**2 / np.sum(singular_values**2)

plt.figure(figsize=(8, 5))
plt.scatter(projection[:len(X), 0], projection[:len(X), 1], label="Unhealthy", alpha=0.8)
plt.scatter(projection[len(X):, 0], projection[len(X):, 1], label="Healthy", alpha=0.8)
plt.xlabel(f"PC1 ({variance[0] * 100:.1f}% variance)")
plt.ylabel(f"PC2 ({variance[1] * 100:.1f}% variance)")
plt.title("PCA overview of extracted features")
plt.legend()
plt.grid(alpha=0.25)
plt.show()
print(f"PCA variance explained: PC1={variance[0]:.3f}, PC2={variance[1]:.3f}")

## Train and validate

Participant-held-out validation on unhealthy labeled recordings.


In [ ]:
def impute_scale(train, test):
    medians = np.zeros(train.shape[1])
    for index in range(train.shape[1]):
        valid = train[:, index][np.isfinite(train[:, index])]
        medians[index] = np.median(valid) if len(valid) else 0.0
    train = np.where(np.isfinite(train), train, medians)
    test = np.where(np.isfinite(test), test, medians)
    mean, scale = train.mean(0), train.std(0)
    scale[scale < 1e-12] = 1.0
    return (train - mean) / scale, (test - mean) / scale

def fit_ridge(train, target, alpha=10.0):
    design = np.column_stack([np.ones(len(train)), train])
    penalty = alpha * np.eye(design.shape[1])
    penalty[0, 0] = 0
    return np.linalg.solve(design.T @ design + penalty, design.T @ target)

def predict_ridge(weights, features):
    return np.clip(np.column_stack([np.ones(len(features)), features]) @ weights, 0, 4)

def ridge_predict(train, target, test, alpha=10.0):
    train, test = impute_scale(train, test)
    return predict_ridge(fit_ridge(train, target, alpha), test)

def mae(actual, predicted):
    return float(np.mean(np.abs(actual - predicted)))

def rmse(actual, predicted):
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))

# Internal check: hold out each unhealthy participant without augmentation.
validation = []
for held_out in sorted(set(groups)):
    train_mask, test_mask = groups != held_out, groups == held_out
    prediction = ridge_predict(X[train_mask], y[train_mask], X[test_mask])
    validation.extend(zip([held_out] * test_mask.sum(), y[test_mask], prediction))
validation_actual = np.array([row[1] for row in validation])
validation_predicted = np.array([row[2] for row in validation])
print("Unhealthy participant-held-out check:", f"MAE={mae(validation_actual, validation_predicted):.3f}", f"RMSE={rmse(validation_actual, validation_predicted):.3f}")


## Test on healthy recordings

Fit on all unhealthy labeled rows, then predict the healthy rows.


In [ ]:
# Final model: fit on all unhealthy clinician-scored rows, then test on healthy rows.
healthy_prediction = ridge_predict(X, y, X_healthy)
assert len(X) == 16 and len(X_healthy) == 12
assert len(healthy_prediction) == len(healthy)
print("Training rows (unhealthy with clinician score):", len(X))
print("Testing rows (healthy):", len(X_healthy))
print("Healthy rows are excluded from training and have no ground-truth labels.")
print("Healthy predictions: participant | task | continuous_0_to_4 | rounded_score")
for row, prediction in zip(healthy, healthy_prediction):
    print(f"{row['participant_id']:8s} | {row['task']:24s} | {prediction:.2f} | {round(prediction):d}")
print("\nHealthy summary")
for participant in sorted({row['participant_id'] for row in healthy}):
    mask = np.array([row['participant_id'] == participant for row in healthy])
    print(f"{participant}: mean={healthy_prediction[mask].mean():.2f}, mean rounded={np.rint(healthy_prediction[mask]).mean():.2f}")


## Train versus test comparison

Descriptive feature and prediction comparisons.


In [ ]:
import matplotlib.pyplot as plt

selected = [("signal_std", "Signal variability"), ("tremor_power_ratio", "Tremor ratio"), ("tap_rate_hz", "Tapping rate")]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for axis, (feature, label) in zip(axes, selected):
    index = numeric_names.index(feature)
    axis.boxplot([X[:, index][np.isfinite(X[:, index])], X_healthy[:, index][np.isfinite(X_healthy[:, index])]], tick_labels=["Unhealthy", "Healthy"])
    axis.set_title(label)
    axis.grid(axis="y", alpha=0.25)
fig.suptitle("Selected feature comparison")
fig.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.boxplot([y, healthy_prediction], tick_labels=["Unhealthy train scores", "Healthy predictions"])
plt.ylim(0, 4)
plt.ylabel("Score (0–4)")
plt.title("Unhealthy training scores versus healthy predictions")
plt.grid(axis="y", alpha=0.25)
plt.show()
print("Healthy predictions have no clinician ground truth for accuracy testing.")

## Summary

- The final model uses no augmentation.
- Healthy scores are predictions only because healthy clinician scores are unavailable.
- The dataset is small and the task-to-score mapping needs clinician confirmation.
